# Economic Exposure: reading GENOME through trade and dependency

## Purpose of this notebook
GENOME can currently be navigated **by country, by organisation and by leader**.
This notebook proposes a fourth angle: **economic exposure**. Instead of looking at
a single actor, it looks at **pairs of countries** and places their political
relationship (from GENOME) next to how economically intertwined they are (from
Eurostat trade data).

The core idea: a conflictual relationship between two countries that barely trade
means something very different from the same conflict between countries that
depend on each other for gas, oil or chips. Policymakers mainly need to see the
tensions where economic exposure is high.

Questions this angle helps answer:
- Which relationships are **politically tense but economically intertwined**, and therefore the most sensitive?
- Are countries **milder towards suppliers they depend on** (e.g. for gas)?
- Does a deteriorating political relationship go together with a shift in trade,
  and does politics lead or follow?




In [ ]:
from data_events import data_genome
from data_trading import data_trade

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import os
from dotenv import load_dotenv

import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()                                        # leest .env als die bestaat
API_KEY = os.environ.get("GENOME_API_KEY")

if not API_KEY:                                      # geen .env? dan vragen
    API_KEY = getpass("GENOME API-key (Enter om over te slaan): ") or None

print("Key ingesteld:", API_KEY is not None)



**From GENOME** (`data_genome`): directed, A = actor, B = recipient
| Array | Meaning |
|---|---|
| `ev_count` | number of events from A to B every month|
| `ev_conflict` / `ev_coop` | idem, split into conflict and cooperation every month |
| `ev_intensity_sum` | sum of the intensity of those events (two events of +0.3 and +0.5 = +0.8) every month |
| `ev_intensity_mean` | average intensity per event (NaN if there were no events) every month |
| `ev_count_period`, `ev_intensity_period` | `ev_count` and the summed intensity over the whole period, shape `[C, D]` |

**From Eurostat** (`data_trade`): all values in euro and monthly shown
| Array | Meaning |
|---|---|
| `tr_flow` | value of goods flowing **from A to B** (A's exports to B) |
| `tr_total` | total trade between A and B: `tr_flow[A,B] + tr_flow[B,A]` |
| `tr_net` | net trade: `tr_flow[A,B] - tr_flow[B,A]` (positive = A has a surplus with B) |
| `tr_balance` | normalised trade balance: `tr_net / tr_total`, between -1 (only imports) and +1 (only exports) |

For specific commodities the product is added as a suffix, e.g. `tr_flow_gas`, `tr_balance_gas` (HS 2711).

## Settings
Choose the **time frame** and the **countries** you want to compare.

- **Time frame:** `start` and `eind` in the format `"YYYY-MM"`. Both months are included.
  Make sure the period falls within GENOME's coverage (check `/events/time-range`)
  and that Eurostat already has data for it (Eurostat lags a few months).
- **Countries:** two-letter ISO codes (e.g. `"FR"`, `"CN"`).
  - **EU countries** can be compared with each other and with any non-EU country.
  - **Non-EU countries** can only be compared with EU countries. Eurostat only
    contains trade reported by EU member states, so there is no trade data
    between two non-EU countries (e.g. China–Russia). These pairs remain empty (NaN).
- **Product** (for `data_trade`): `"TOTAL"` for all goods, or an HS code such as
  `"2711"` (gas), `"2709"` (crude oil) or `"8542"` (chips).

In [ ]:
# ---------- settings ----------
start, eind = "2023-01", "2025-02"          # time frame (YYYY-MM), both months included

# EU countries can be compared with each other and with non-EU countries.
# Non-EU countries can only be compared with EU countries (no Eurostat data between two non-EU countries).
landen = ["FR", "DE", "IT",                 # EU
          "SA", "CN", "RU", "DZ","NL","HU"]           # non-EU

maanden = pd.period_range(start, eind, freq="M").strftime("%Y-%m").tolist()

#"TOTAL" for all goods, or an HS code such as "2711" (gas), "2709" (crude oil) or "8542" (chips).
product = "TOTAL"


# ---------- retrieve data ----------
(ev_count_period, ev_intensity_period, ev_intensity_sum, ev_intensity_mean,
 ev_count, ev_conflict, ev_coop) = data_genome(API_KEY, landen, start, eind)

tr_flow, tr_balance, tr_net, tr_total = data_trade(product, landen, start, eind)

## Relationship and trade over time for one country pair

This chart follows a single country pair month by month and places the
**political relationship** next to the **volume of trade**.

- **Green line (left axis): cumulative intensity.** The GENOME intensity of all
  events between A and B (both directions) is summed per month and then added
  up over time. A rising line means the relationship is accumulating cooperation,
  a falling line means it is accumulating conflict. The slope shows how fast the
  relationship changes in a given month.
- **Blue bars (right axis): total trade** between A and B per month, in billion euro
  (Eurostat).

**How to use it:** change `A` and `B` to any pair in `landen`. Note that at least
one of the two must be an EU country, otherwise there is no trade data.

**How to read it:** look at whether shifts in the relationship coincide with shifts
in trade, for example a sharp drop in intensity followed by declining

In [ ]:
A, B = "FR", "RU"







i, j = landen.index(A), landen.index(B)
x = pd.to_datetime(maanden)

intensity_cum = np.cumsum(ev_intensity_sum[:, i, j] + ev_intensity_sum[:, j, i])
trade = tr_total[:, i, j] / 1e9                     # billion euro

fig, ax1 = plt.subplots(figsize=(12, 5))

# trade as bars on the right axis (background)
ax2 = ax1.twinx()
ax2.bar(x, trade, width=20, color="#3b6fd8", alpha=0.35, label="Total trade (bn euro)")
ax2.set_ylabel("Total trade (bn euro)")
ax2.set_ylim(0, np.nanmax(trade) * 1.15)

# intensity as a line on the left axis (foreground)
ax1.axhline(0, color="grey", lw=0.8)
ax1.plot(x, intensity_cum, color="#2fb98f", marker="o", lw=2.5, label="Intensity (cumulative)")
ax1.set_ylabel("Intensity (cumulative)")
ax1.set_zorder(ax2.get_zorder() + 1)                # line above the bars
ax1.patch.set_visible(False)

# one legend for both
l1, n1 = ax1.get_legend_handles_labels()
l2, n2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, n1 + n2, loc="upper left", frameon=False)

ax1.set_title(f"{A} \u2194 {B}: relationship and trade per month", loc="left")
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

plt.tight_layout()
plt.show()

## Relationship vs. trade balance per country pair

This bubble chart summarises the **whole period** for every country pair in one view.

- **x-axis: average relationship.** The mean GENOME intensity per event between
  A and B, both directions combined. Left of zero the relationship is dominated
  by conflict, right of zero by cooperation.
- **y-axis: trade balance** of A with B over the whole period:
  (exports − imports) / (exports + imports). Above zero A has a trade surplus with B,
  below zero a deficit.
- **Bubble size: total trade** between A and B over the period (billion euro).
- **Colour:** green for a cooperative, red for a conflictual relationship.

Each pair is shown once, always **from the perspective of the EU country** (if both
are EU members, the one listed first in `landen`). Pairs of two non-EU countries
are left out, because Eurostat has no trade data for them.

**How to read it:** the most policy-relevant pairs are **large red bubbles**:
substantial trade combined with a conflictual relationship. Those in the lower-left
quadrant are the most sensitive: the EU country *imports* more than it exports
from a partner it is in conflict with, which points to dependency.

**Method note:** the average relationship and the trade balance are computed from
**totals over the period** (sum of intensity ÷ number of events, and sum of net
trade ÷ sum of total trade) rather than by averaging the monthly values. This way
months with more events or more trade weigh more heavily, instead of every month
counting equally.

In [ ]:
# ---------- bubble chart: relationship vs trade balance, bubble size = total trade ----------
# Uses: landen, maanden, ev_count_period, ev_intensity_period ([A, B])
#       tr_total, tr_net ([month, A, B])


EU = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE",
      "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"]

# ---------- 1. totals over the period -> (N, N) ----------
# relationship: both directions combined
n_events = ev_count_period + ev_count_period.T
intensity = ev_intensity_period + ev_intensity_period.T

# trade: sum over months, then divide
trade_total = np.nansum(tr_total, axis=0)
trade_net = np.nansum(tr_net, axis=0)

with np.errstate(invalid="ignore", divide="ignore"):
    mean_relation = intensity / n_events              # average intensity per event
    balance_period = trade_net / trade_total          # trade balance over the period

# ---------- 2. each pair once, from the EU country's perspective ----------
pairs = []
for a in range(len(landen)):
    for b in range(a + 1, len(landen)):
        # A = EU country (if both are EU: the first one in the list)
        a_, b_ = (b, a) if (landen[a] not in EU and landen[b] in EU) else (a, b)
        if trade_total[a_, b_] > 0 and not np.isnan(mean_relation[a_, b_]):
            pairs.append((a_, b_))

ia = np.array([p[0] for p in pairs])
ib = np.array([p[1] for p in pairs])
x = mean_relation[ia, ib]
y = balance_period[ia, ib]
size = trade_total[ia, ib] / 1e9                      # billion euro

# ---------- 3. figure ----------
fig, ax = plt.subplots(figsize=(11, 7))
ax.axvline(0, color="grey", lw=0.8)
ax.axhline(0, color="grey", lw=0.8)

s = 2000 * size / np.nanmax(size)                     # bubble area ~ trade
colour = np.where(x >= 0, "#2fb98f", "#e5484d")       # green = cooperation, red = conflict
ax.scatter(x, y, s=s, c=colour, alpha=0.55, edgecolors="white", linewidths=1.5, zorder=3)

for k in range(len(x)):
    ax.annotate(f"{landen[ia[k]]}\u2013{landen[ib[k]]}", (x[k], y[k]),
                ha="center", va="center", fontsize=8, zorder=4)

# legend for bubble size
for value in [np.nanmax(size) / 10, np.nanmax(size) / 2, np.nanmax(size)]:
    ax.scatter([], [], s=2000 * value / np.nanmax(size), c="lightgrey",
               edgecolors="grey", label=f"{value:.0f} bn euro")
ax.legend(title="Total trade", loc="center left", bbox_to_anchor=(1.01, 0.5),
          frameon=False, labelspacing=2.5, borderpad=1.5)

xmax = max(0.05, np.nanmax(np.abs(x))) * 1.3
ax.set_xlim(-xmax, xmax)
ax.set_ylim(-1.05, 1.05)
ax.set_xlabel("Average relationship (GENOME intensity per event)   \u2190 conflict  |  cooperation \u2192")
ax.set_ylabel("Trade balance from the first country's perspective\n\u2190 deficit  |  surplus \u2192")
ax.set_title(f"Relationship, trade balance and trade volume per country pair, "
             f"{maanden[0]} to {maanden[-1]}", loc="left")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## Commodity lens: gas (HS 2711)

The analysis so far used **total trade** between countries. Here the same trade data
is retrieved for a single strategic commodity: **gas**. Gas is one of the clearest
examples of geopolitical dependency: supply is concentrated in a few producers,
it is hard to replace in the short term (pipelines, LNG terminals), and it has been
used as political leverage, most visibly in the relationship between the EU and Russia.

The product code is **HS 2711 – petroleum gases**, which covers:
- natural gas via **pipeline** (HS 271121)
- **LNG**, liquefied natural gas transported by ship (HS 271111)
- **LPG** such as propane and butane (smaller share)

The resulting arrays have the same structure as for total trade (`[month, A, B]`),
with the suffix `_gas`:

| Array | Meaning |
|---|---|
| `tr_flow_gas` | gas flowing from A to B |
| `tr_total_gas` | total gas trade between A and B |
| `tr_net_gas` | net gas trade (positive = A exports more gas to B than it imports) |
| `tr_balance_gas` | normalised gas balance, between -1 (only imports) and +1 (only exports) |

**What to expect:** for pairs of an EU country and a gas producer (Russia, Algeria) the balance will be close to **-1**: the EU country imports gas but hardly exports any back. The interesting information is therefore mainly in the
**volume** of gas trade and how it relates to the political relationship.

**Caveats**
- Values are in **euro**, so they also reflect gas prices, which fluctuated strongly
  in this period. Changes in value do not necessarily mean changes in volume.


In [ ]:
tr_flow_gas, tr_balance_gas, tr_net_gas, tr_total_gas = data_trade("2711", landen, start, eind)

In [ ]:
# ---------- bubble chart gas: relationship vs gas balance, bubble size = total gas trade ----------
# Uses: landen, maanden, ev_count_period, ev_intensity_period ([A, B])
#       tr_total_gas, tr_net_gas ([month, A, B]) from data_trade("2711", ...)

import numpy as np
import matplotlib.pyplot as plt


MIN_SHARE = 0.01 


# ---------- settings ----------
UNIT = "euro"           # "euro" for VALUE_EUR, "kg" for QUANTITY_KG
DIVISOR, LABEL = (1e9, "Mt") if UNIT == "kg" else (1e9, "bn euro")

EU = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE",
      "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"]

# ---------- 1. totals over the period -> (N, N) ----------
n_events = ev_count_period + ev_count_period.T          # both directions combined
intensity = ev_intensity_period + ev_intensity_period.T

gas_total = np.nansum(tr_total_gas, axis=0)             # sum over months, then divide
gas_net = np.nansum(tr_net_gas, axis=0)

with np.errstate(invalid="ignore", divide="ignore"):
    mean_relation = intensity / n_events                # average intensity per event
    gas_balance = gas_net / gas_total                   # -1 = only imports, +1 = only exports

# ---------- 2. each pair once, from the EU country's perspective ----------
pairs = []
for a in range(len(landen)):
    for b in range(a + 1, len(landen)):
        # A = EU country (if both are EU: the first one in the list)
        a_, b_ = (b, a) if (landen[a] not in EU and landen[b] in EU) else (a, b)
        if gas_total[a_, b_] > 0 and not np.isnan(mean_relation[a_, b_]):
            pairs.append((a_, b_))                      # skip pairs without events or gas trade

ia = np.array([p[0] for p in pairs])
ib = np.array([p[1] for p in pairs])
x = mean_relation[ia, ib]
y = gas_balance[ia, ib]
size = gas_total[ia, ib] / DIVISOR

keep = size >= MIN_SHARE * np.nanmax(size)
ia, ib, x, y, size = ia[keep], ib[keep], x[keep], y[keep], size[keep]

# ---------- 3. figure ----------
fig, ax = plt.subplots(figsize=(11, 7))
ax.axvline(0, color="grey", lw=0.8)
ax.axhline(0, color="grey", lw=0.8)

s = 2000 * size / np.nanmax(size)                       # bubble area ~ gas trade
colour = np.where(x >= 0, "#2fb98f", "#e5484d")         # green = cooperation, red = conflict
ax.scatter(x, y, s=s, c=colour, alpha=0.55, edgecolors="white", linewidths=1.5, zorder=3)

for k in range(len(x)):
    ax.annotate(f"{landen[ia[k]]}\u2013{landen[ib[k]]}", (x[k], y[k]),
                ha="center", va="center", fontsize=8, zorder=4)

# legend for bubble size
for value in [np.nanmax(size) / 10, np.nanmax(size) / 2, np.nanmax(size)]:
    ax.scatter([], [], s=2000 * value / np.nanmax(size), c="lightgrey",
               edgecolors="grey", label=f"{value:.2g} {LABEL}")
ax.legend(title="Total gas trade", loc="center left", bbox_to_anchor=(1.01, 0.5),
          frameon=False, labelspacing=2.5, borderpad=1.5)

xmax = max(0.05, np.nanmax(np.abs(x))) * 1.3
ax.set_xlim(-xmax, xmax)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("Average relationship (GENOME intensity per event)   \u2190 conflict  |  cooperation \u2192")
ax.set_ylabel("Gas balance of the EU country\n\u2190 net imports  |  net exports \u2192")
ax.set_title(f"Relationship and gas trade (HS 2711) per country pair, "
             f"{maanden[0]} to {maanden[-1]}", loc="left")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

